###Ingest races.csv file
        1. Read the file using spark dataframe reader API
        2. Add Metadata Columns
            - Source File
            - Ingestion Timestamp
        3. Write to bronze delta table

####Step 1 - Read the CSV file using the dataframe reader API

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType

races_schema = StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType())
    ])


In [0]:
%python
races_df = (
    spark.read
        .format('csv')
        .option('header', True)
#        .option('inferSchema', True)
        .option('mode', 'FAILFAST')
        .schema(races_schema)
        .load('/Volumes/formula1/landing/files/races.csv')
)

In [0]:
%python
display(races_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
%python
from pyspark.sql import functions as F

races_final_df = (
    races_df
        .withColumn('ingestion_timestamp', F.current_timestamp())
        .withColumn('source_file', F.col('_metadata.file_path')))

In [0]:
%python

display(races_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
%python
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('formula1.bronze.circuits')
)

In [0]:
%sql
SELECT * FROM formula1.bronze.circuits

In [0]:
%python

display(spark.table('formula1.bronze.circuits'))